# MVP — Step-by-Step Text-to-SQL Pipeline

A minimal, linear walkthrough of how the agent answers a business question.

| Step | What happens |
|------|-------------|
| 1 | Business question is asked |
| 2 | Connect to the database |
| 3 | Inspect the DB schema |
| 4 | Profile the data |
| 5 | Discover table relationships |
| 6 | Retrieve relevant schema docs (RAG) |
| 7 | Generate SQL (LLM) |
| 8 | Validate the SQL |
| 9 | Execute the SQL |
| 10 | Analyze results → natural language answer |
| 11 | (Optional) chart visualization |

In [ ]:
import os, sys, json
from pathlib import Path
from sqlalchemy import text
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

---
## Step 1 — Business question

A stakeholder asks a question in plain English:

In [ ]:
BUSINESS_QUESTION = "How many orders have been delivered?"
print(f"Question: {BUSINESS_QUESTION}")

---
## Step 2 — Connect to the database

Create a SQLAlchemy engine and verify the connection works.

In [ ]:
from src.database.connection import create_database_engine, verify_connection

engine = create_database_engine('sqlite:///data/olist.db')
print(f'Connected: {verify_connection(engine)}')

---
## Step 3 — Show DB schema

What tables exist, and what columns does each one have?

In [ ]:
from src.schema.extractor import SchemaExtractor

extractor = SchemaExtractor(engine)
schema = extractor.extract()

for t in schema.tables:
    cols = ', '.join([c.name for c in t.columns])
    print(f'{t.name:45s} ({len(t.columns)} cols)  {cols}')

Show columns + types for the `orders` table specifically:

In [ ]:
orders = next(t for t in schema.tables if t.name == 'orders')

print(f'Table: {orders.name}')
print(f'Primary keys: {orders.primary_keys}')
print()
for c in orders.columns:
    print(f'  {c.name:35s}  {c.data_type:10s}  nullable={c.nullable}')

print(f'\nForeign keys:')
for fk in orders.foreign_keys:
    print(f'  {fk.constrained_column} -> {fk.referred_table}.{fk.referred_column}')

---
## Step 4 — Profile the data

Get row counts, null percentages, unique counts, and sample values for every column.

In [ ]:
from src.schema.profiler import DatabaseProfiler

profiler = DatabaseProfiler(engine, schema, sample_size=500)
profile = profiler.profile_all()

# Show the orders table profile
orders_profile = next(t for t in profile['tables'] if t['table'] == 'orders')
print(f"Table: {orders_profile['table']}  ({orders_profile['row_count']} rows)")
print()
for col_name, stats in orders_profile['columns'].items():
    print(f"  {col_name:35s}  type={stats['type']:12s}  nulls={stats['null_count']:6d}  unique={stats['unique_count']:6d}")

---
## Step 5 — Discover relationships

Which tables are linked, and how?

In [ ]:
from src.schema.relationships import RelationshipDiscoverer

discoverer = RelationshipDiscoverer(engine, schema, profile)
relationships = discoverer.discover_all()

print(f'{len(relationships)} relationships found:\n')
for r in relationships:
    print(f'  {r.source_table}.{r.source_column:25s} -> {r.target_table}.{r.target_column:25s}  '
          f'{r.relationship_type:15s}  conf={r.confidence:.3f}  ({r.source})')

---
## Step 6 — Retrieve relevant schema docs (RAG)

Instead of feeding the entire schema to the LLM, embed all table/column descriptions
into a vector index and retrieve only the top-k most relevant ones for this question.

In [ ]:
from src.retrieval.retriever import SchemaRetriever

retriever = SchemaRetriever('data/semantic_schema.json')
retrieved = retriever.retrieve(BUSINESS_QUESTION, top_k=5)

print(f'Retrieved {len(retrieved)} schema docs for:\n  "{BUSINESS_QUESTION}"\n')
for doc, score in retrieved:
    print(f'  [{score:.3f}] {doc["content"]}')

---
## Step 7 — Generate SQL (LLM)

The question + retrieved schema + relationships are sent to the LLM, which returns
a JSON object with the SQL query and its reasoning.

> Requires `LLM_API_KEY` env var. The cell below will skip this step if not set.

In [ ]:
from src.agent.sql_generator import SQLGenerator
from src.agent.llm_client import LLMClient

api_key = os.environ.get('LLM_API_KEY', '').strip()
has_key = bool(api_key) and api_key != 'your-api-key-here'

if has_key:
    llm = LLMClient()
    generator = SQLGenerator(llm)
    gen_result = generator.generate(BUSINESS_QUESTION, retrieved, relationships)
    sql = gen_result.get('sql', '')
    print(f'Generated SQL:\n{sql}')
    print(f'\nReasoning: {gen_result.get("reasoning_summary", "")}')
    print(f'Assumptions: {gen_result.get("assumptions", "")}')
else:
    # Fallback: hard-coded SQL for demo without LLM
    sql = "SELECT COUNT(*) AS order_count FROM orders WHERE order_status = 'delivered'"
    print(f'No LLM key — using fallback SQL:\n{sql}')

---
## Step 8 — Validate the SQL

Check syntax, ensure it's a SELECT, and reject banned keywords.

In [ ]:
from src.agent.validator import SQLValidator

validator = SQLValidator()
is_valid, error = validator.validate(sql)

if is_valid:
    print('SQL is valid.')
else:
    print(f'Validation failed: {error}')

---
## Step 9 — Execute the SQL

In [ ]:
with engine.connect() as conn:
    result_set = conn.execute(text(sql))
    rows = [dict(row._mapping) for row in result_set]

print('Results:')
pd.DataFrame(rows)

---
## Step 10 — Analyze results → natural language answer

The LLM summarizes the raw SQL results into a human-readable answer.

> Requires `LLM_API_KEY`. Falls back to a simple summary without it.

In [ ]:
if has_key:
    from src.agent.analyst import ResultAnalyst
    analyst = ResultAnalyst(llm)
    answer = analyst.analyze(BUSINESS_QUESTION, sql, rows)
    print(f'Answer:\n{answer}')
else:
    # Simple fallback
    count = rows[0].get('order_count', rows[0].get(list(rows[0].keys())[0], '?'))
    answer = f"There are {count:,} delivered orders in the database."
    print(f'Answer (no LLM):\n{answer}')

---
## Step 11 — (Optional) Chart visualization

If the results are better shown as a chart, the visualizer generates a Plotly spec.

In [ ]:
# Demo with a query that produces chartable data
chart_sql = """
    SELECT order_status, COUNT(*) AS order_count
    FROM orders
    GROUP BY order_status
    ORDER BY order_count DESC
"""

with engine.connect() as conn:
    chart_rows = [dict(r._mapping) for r in conn.execute(text(chart_sql))]

chart_df = pd.DataFrame(chart_rows)
print('Data for charting:')
print(chart_df.to_string(index=False))
print()

# Plot directly with matplotlib (no LLM needed)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
chart_df.plot.bar(x='order_status', y='order_count', ax=ax, legend=False, color='steelblue')
ax.set_title('Orders by Status')
ax.set_ylabel('Count')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## Summary

```
Business Question
       │
       ▼
  ┌─────────────┐
  │  Step 2-5   │  Extract schema, profile data, find relationships
  │  (offline)  │  → data/schema.json, data/database_profile.json, data/relationships.json
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │  Step 6     │  RAG: retrieve relevant schema docs for this question
  │  (FAISS)    │
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │  Step 7     │  LLM generates SQL from question + schema + relationships
  │  (LLM)      │
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │  Step 8     │  Validate: sqlglot syntax + SELECT-only + no banned keywords
  │  (sqlglot)  │  If invalid → repair via LLM (up to 3 attempts)
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │  Step 9     │  Execute SQL against the database
  │  (SQLAlchemy)│
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │  Step 10    │  LLM summarizes results into a natural language answer
  │  (LLM)      │
  └──────┬──────┘
         ▼
     Answer + optional chart
```

Steps 2–5 only run once (or when the schema changes). Steps 6–10 run per question.